# Spine preprocessing - visual check

Runs the full preprocessing pipeline (all stages through SDF sample generation - same stages and config a real run would use, just filtered to the first `N` spines of one branch, see CLAUDE.md "mac = testing, Windows = full datasets") and renders, per spine, up to seven 3D checks (`docs/neuron-model/s-module-preprocessing.md`):

0. the source mesh;
1. the sealed mesh - faces added while sealing in **green**, the attachment loop (the boundary that got capped) outlined in **red**;
1b. the sealed mesh again, but with *every* hole sealing filled - not just the attachment region - each hole its own solid color, no attachment-loop overlay. This is the one to look at for `attachment_ambiguous:close_candidates`/`attachment_ambiguous:attachment_loop_not_closed` spines (figure "1" has no loop to outline for those - there is no chosen candidate);
2. the sealed mesh in local coordinates - attachment cap faces in **red**, origin + tangent/radial/binormal axes drawn;
3. the source mesh in local coordinates (same transform, no independent origin/axes recompute - `s-module-preprocessing.md` 3.3), origin + axes drawn;
4. the sampled point clouds - one 3D scatter per (point count, seed variant) combination, attachment-cap points in red;
5. the SDF query pool - surface/near-surface/uniform samples colored by signed distance (negative = inside), over a translucent copy of the local sealed mesh.

Plotting helpers live in `src/neuron_model/visualization.py`; this notebook only picks spines, runs stages, and calls them.

In [9]:
%load_ext autoreload
%autoreload 2
# Reloads modules under src/ automatically before running each cell, so
# edits to .py files show up without restarting the kernel. Caveats: it
# won't pick up changes to a dataclass's fields for objects already
# constructed (e.g. an existing `cfg`), or `from x import y` bindings
# captured before the edit (re-run the import cell after such changes) -
# see https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html

from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Repository root was not found.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.neuron_model.spine_preprocessing import (
    SpinePreprocessingConfig,
    discover_spines,
    run_full_spine_preprocessing_pipeline,
)
from src.neuron_model.visualization import visualize_spine_stage_outputs

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

Selects the first `N` spines of one branch by `neuron_id`/`limb_id`/`branch_id` + `limit`.

In [10]:
N = 2  # number of spines to test on (first N, by filename order)
DATASET = "minnie65"
NEURON_ID = "864691134886335738"
# LIMB_ID = "001"
# BRANCH_ID = "002"
LIMB_ID = "001"
BRANCH_ID = "002"

# Keeps this notebook's output out of datasets/ (raw data) and out of git
# (data/ is gitignored) - see docs/project-architecture.md "data/interim/".
OUTPUT_ROOT = REPO_ROOT / "data" / "interim" / "neuron-model" / "visual-check"

cfg = SpinePreprocessingConfig.from_yaml(
    REPO_ROOT / "data" / "processed" / "config" / "spine_preprocessing.yaml",
    raw_data_root=REPO_ROOT / "datasets",
    # Anchored to REPO_ROOT, not left as the SpinePreprocessingConfig default
    # ("datasets", resolved against the process CWD) - Jupyter's CWD is often
    # the notebook's own directory, not the repo root, which silently
    # resolves to a nonexistent path and leaves cell_type/compartment null.
    metadata_root=REPO_ROOT / "datasets",
    output_root=OUTPUT_ROOT,
    dataset=DATASET,
    neuron_id=NEURON_ID,
    limb_id=LIMB_ID,
    branch_id=BRANCH_ID,
    limit=N,
    workers=1,       # mac smoke-test scale; see CLAUDE.md "mac = testing, Windows = full datasets"
    force=True,       # rerun cleanly every time this notebook is executed
    # allow_needs_review defaults to True (s-module-preprocessing.md 3.5.3):
    # a needs_review spine (e.g. self_intersections) still gets the full
    # pipeline by default, so this notebook doesn't need to override it.
)

records = discover_spines(cfg)
print(f"Selected {len(records)} spine(s):")
for record in records:
    print(" ", record.object_id, "->", record.source_path)

Selected 2 spine(s):
  minnie65/864691134886335738/001/002/000 -> /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65/864691134886335738/limb_001/branch_002/spines/spine_000.off
  minnie65/864691134886335738/001/002/001 -> /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65/864691134886335738/limb_001/branch_002/spines/spine_001.off


## Run the full pipeline

`run_full_spine_preprocessing_pipeline()` - the same entry point a real run uses (seal -> false-spine -> orient -> qc -> pointcloud -> sdf -> manifest), just filtered to `records` (the first `N` spines) via `cfg`. Every result file listed in `docs/neuron-model/s-module-preprocessing.md` §4 gets produced, including `manifest.parquet`/`metadata.json`, not only the ones this notebook plots.

In [11]:
pipeline_result = run_full_spine_preprocessing_pipeline(cfg)
for stage, frame in pipeline_result.items():
    print(stage, frame["status"].value_counts(dropna=False).to_dict())

seal [visual-check]:   0%|          | 0/2 [00:00<?, ?it/s]


[seal] /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65
  Processed:           2
  success              2


false-spine [visual-check]:   0%|          | 0/2 [00:00<?, ?it/s]

Number of vertices of the skeleton: 93
Number of edges of the skeleton: 92
Number of vertices of the skeleton: 37
Number of edges of the skeleton: 36

[false-spine] /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65
  Processed:           2
  invalid              1
  valid                1


orient [visual-check]:   0%|          | 0/1 [00:00<?, ?it/s]

...

morphometrics [visual-check]:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/eklipsss/Documents/GitHub/spinetool/src/spine_analysis/mesh/utils.py:22: RuntimeWarning: invalid value encountered in cast
  facets = np.ndarray((mesh.size_of_facets(), 3)).astype("uint")



[morphometrics] /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65
  Processed:           2
  skipped              1
  success              1


metadata [visual-check]:   0%|          | 0/2 [00:00<?, ?it/s]


[metadata] /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65
  Processed:           2
  success              2

[manifest] /Users/eklipsss/Documents/GitHub/spinetool/datasets/minnie65
  Processed:           2
  invalid              1
  valid                1
seal {'success': 2}
false_spine_detection {'invalid': 1, 'valid': 1}
orient {'skipped': 1, 'success': 1}
mesh_qc {'skipped': 1, 'valid': 1}
pointcloud {'skipped': 1, 'success': 1}
sdf {'skipped': 1, 'success': 1}
morphometrics {'skipped': 1, 'success': 1}
metadata {'success': 2}
manifest {'invalid': 1, 'valid': 1}


## Visual QA plots

Up to seven figures per spine: 0) original, 1) sealed (new faces green / attachment loop red), 1b) all sealed holes, one color each, no attachment marking (skips straight past the "no loop chosen" case that leaves figure 1 without a red outline), 2) local sealed (attachment cap red + local frame), 3) local original (local frame), 4) point clouds grid (rows = point count, columns = seed variant; red = attachment cap), 5) SDF samples (colored by signed distance; negative = inside).

In [12]:
figures_by_spine = {}
for record in records:
    print(f"\n=== spine {record.object_id} ===")
    figures_by_spine[record.spine_id] = visualize_spine_stage_outputs(record, cfg)


=== spine minnie65/864691134886335738/001/002/000 ===
spine 000: skipping 2) local sealed mesh - file not found (not produced for this spine): /Users/eklipsss/Documents/GitHub/spinetool/data/interim/neuron-model/visual-check/minnie65/864691134886335738/limb_001/branch_002/spines/spine_000/local_sealed_spine_000.off
spine 000: skipping 3) local original mesh - file not found (not produced for this spine): /Users/eklipsss/Documents/GitHub/spinetool/data/interim/neuron-model/visual-check/minnie65/864691134886335738/limb_001/branch_002/spines/spine_000/local_spine_000.off
spine 000: skipping 4) point clouds - file not found (not produced for this spine): /Users/eklipsss/Documents/GitHub/spinetool/data/interim/neuron-model/visual-check/minnie65/864691134886335738/limb_001/branch_002/spines/spine_000/pointcloud_2048_1.npz
spine 000: skipping 5) SDF samples - file not found (not produced for this spine): /Users/eklipsss/Documents/GitHub/spinetool/data/interim/neuron-model/visual-check/minnie


=== spine minnie65/864691134886335738/001/002/001 ===
